# Pretrain Commutative CNN Encoder

Load the shared unlabeled pretraining dataset and save commutative CNN encoder weights for downstream classification notebooks.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from sklearn.model_selection import train_test_split

from src.ml import (
    CommutativeCNNClassifier,
    CommutativeCNNConfig,
    CommutativeCNNPretrainingConfig,
    LossWeightConfig,
    OptimizationConfig,
    augment_training_tensors_with_rotations,
    load_commutative_cnn_pretraining_config,
    write_commutative_cnn_pretraining_config,
)
from src.tensor_utils import load_unlabeled_tensor_dataset


In [ ]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
pretrained_encoder_path = Path("artifacts/pretrained_commutative_cnn/encoder_state_v7.pt")
validation_fraction = 0.15
train_num_random_rotations = 0
rotation_range_degrees = 0.0

model_config = CommutativeCNNConfig(
    spatial_conv_channels=(16, 32),
    spatial_kernel_size_z=(3, 1),
    spatial_kernel_size_xy=(5, 3),
    spatial_stride_z=(1, 1),
    spatial_stride_xy=(1, 1),
    spatial_pool_kernel_z=(1, 1),
    spatial_pool_kernel_xy=(2, 2),
    spatial_pool_stride_z=(1, 1),
    spatial_pool_stride_xy=(2, 2),
    temporal_st_channels=(48, 64),
    temporal_st_kernel_sizes=(5, 3),
    temporal_ts_channels=(32, 48, 64),
    temporal_ts_kernel_sizes=(7, 5, 3),
    spatial_agg_channels=(32, 64),
    spatial_agg_kernel_size_z=(3, 1),
    spatial_agg_kernel_size_xy=(3, 3),
    spatial_agg_stride_z=(1, 1),
    spatial_agg_stride_xy=(1, 1),
    spatial_agg_pool_kernel_z=(1, 1),
    spatial_agg_pool_kernel_xy=(1, 2),
    spatial_agg_pool_stride_z=(1, 1),
    spatial_agg_pool_stride_xy=(1, 2),
    patch_size_z=1,
    patch_size_xy=16,
    embedding_dim=64,
    num_prototypes=64,
    probe_local_count=32,
    probe_region_grid=(1, 2, 2),
    probe_time_bins=8,
    probe_frequency_bins=4,
    dropout=0.15,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=70,
    learning_rate=2e-4,
    weight_decay=3e-4,
    early_stopping_patience=16,
    early_stopping_min_delta=5e-5,
    early_stopping_start_epoch=30,
    early_stopping_monitor="loss",
    early_stopping_smoothing="median",
    early_stopping_smoothing_window=5,
    training_plot_dir="artifacts/pretrained_commutative_cnn/loss_plots",
    training_plot_every_n_epochs=2,
    training_plot_smoothing_window=5,
    scheduler_patience=5,
    scheduler_factor=0.75,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    lambda_cross=0.10,
    cross_warmup_epochs=12,
    cross_ramp_epochs=32,
    prototype_temperature=0.20,
    prototype_alignment_weight=0.05,
    prototype_warmup_epochs=12,
    prototype_ramp_epochs=32,
    latent_alignment_weight=0.0,
    lambda_align=0.0,
    probe_mask_probability=0.75,
    probe_alpha_local=1.0,
    probe_alpha_region_time=1.0,
    probe_alpha_derivative=1.0,
    probe_alpha_frequency=1.0,
    probe_alpha_correlation=0.25,
)

pretraining_config = CommutativeCNNPretrainingConfig(
    unlabeled_dataset_path=unlabeled_dataset_path,
    pretrained_encoder_path=pretrained_encoder_path,
    validation_fraction=validation_fraction,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
pretraining_config_path = write_commutative_cnn_pretraining_config(pretraining_config)
pretraining_config = load_commutative_cnn_pretraining_config(pretraining_config_path)
print(f"Commutative CNN pretraining config in {pretraining_config_path}")
print(f"Pretraining loss PDFs: {Path(optimization_config.training_plot_dir).resolve()}")
print(f"Pretrained encoder checkpoint target: {pretrained_encoder_path.resolve()}")
pretraining_config


Loaded commutative CNN pretraining config from /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/artifacts/pretrained_commutative_cnn/config.yaml


CommutativeCNNPretrainingConfig(unlabeled_dataset_path=PosixPath('.dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks'), pretrained_encoder_path=PosixPath('artifacts/pretrained_commutative_cnn/encoder_state_v6.pt'), validation_fraction=0.15, train_num_random_rotations=0, rotation_range_degrees=0.0, model_config=CommutativeCNNConfig(spatial_conv_channels=(16, 32), spatial_kernel_size_z=(3, 1), spatial_kernel_size_xy=(5, 3), spatial_stride_z=(1, 1), spatial_stride_xy=(1, 1), spatial_pool_kernel_z=(1, 1), spatial_pool_kernel_xy=(2, 2), spatial_pool_stride_z=(1, 1), spatial_pool_stride_xy=(2, 2), temporal_st_channels=(48, 64), temporal_st_kernel_sizes=(5, 3), temporal_ts_channels=(32, 48, 64), temporal_ts_kernel_sizes=(7, 5, 3), spatial_agg_channels=(32, 64), spatial_agg_kernel_size_z=(3, 1), spatial_agg_kernel_size_xy=(3, 3), spatial_agg_stride_z=(1, 1), spatial_agg_stride_xy=(1, 1), spatial_agg_pool_kernel_z=(1, 1), spatial_agg_pool_kernel_xy=(1, 2), spatial_agg_pool_strid

In [3]:
unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)
train_indices, val_indices = train_test_split(
    range(len(unlabeled_dataset["tensors"])),
    test_size=validation_fraction,
    random_state=optimization_config.random_state,
    shuffle=True,
)
X_train_base = unlabeled_dataset["tensors"][train_indices]
X_val = unlabeled_dataset["tensors"][val_indices]
metadata_train_base = unlabeled_dataset["metadata"].iloc[train_indices].reset_index(drop=True)
metadata_val = unlabeled_dataset["metadata"].iloc[val_indices].reset_index(drop=True)
X_train, _, metadata_train = augment_training_tensors_with_rotations(
    X_train_base,
    [0] * len(X_train_base),
    metadata=metadata_train_base,
    num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)
{
    "all_tensors": unlabeled_dataset["tensors"].shape,
    "all_metadata": unlabeled_dataset["metadata"].shape,
    "train_base_tensors": X_train_base.shape,
    "train_tensors": X_train.shape,
    "val_tensors": X_val.shape,
    "train_base_metadata": metadata_train_base.shape,
    "train_metadata": metadata_train.shape,
    "val_metadata": metadata_val.shape,
}


{'all_tensors': torch.Size([2144, 20, 5, 96, 96]),
 'all_metadata': (2144, 7),
 'train_base_tensors': torch.Size([1822, 20, 5, 96, 96]),
 'train_tensors': torch.Size([1822, 20, 5, 96, 96]),
 'val_tensors': torch.Size([322, 20, 5, 96, 96]),
 'train_base_metadata': (1822, 7),
 'train_metadata': (1822, 7),
 'val_metadata': (322, 7)}

## Output Review

The interrupted run with `learning_rate=4e-4` improved validation loss at epoch 2 (`0.5846`) but then became unstable by epoch 4 (`0.9772`) while training loss kept falling. The saved `v5` loss curves show that self probes learn, but validation is dominated by noisy cross-probe and correlation terms after the cross-weight ramp starts. The completed `v6` run with `learning_rate=2e-4`, `lambda_cross=0.15`, warmup `16`, ramp `28`, and `lambda_align=0.02` selected `best_epoch=042` with smoothed `best_metric=5.3314`, but validation stayed volatile after the cross objective reached full strength. The next run keeps the optimizer settings, removes exact latent MSE alignment (`latent_alignment_weight=0.0`), lowers cross pressure to `lambda_cross=0.10`, and adds conservative prototype alignment (`num_prototypes=64`, `prototype_temperature=0.20`, `prototype_alignment_weight=0.05`, warmup `12`, ramp `32`). This keeps self-probes dominant early, lets teacher-student cross-probes and prototype alignment ramp together, and writes to `encoder_state_v7.pt` so the existing `encoder_state_v6.pt` checkpoint remains available.

In [4]:
%%time
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
model.pretrain(X_train, validation_data=X_val)
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
pretrained_encoder_path


Commutative CNN model: parameters=170,057 trainable=170,057 size=0.65 MB
cols:
    ep=epoch
    lr=learning_rate
    eta=estimated_time_remaining
    trL=train_loss
    trS=train_self_probe_loss
    trX=train_cross_probe_loss
    trFA=train_feature_alignment_loss
     ep       lr       eta |      trL      trS      trX     trFA |      vaL      vaS      vaX     vaFA
001/070 2.00e-04  16:42:52 |   3.2873   3.2798   3.6117   0.3764 |  22.6332  21.8092  11.9942  41.2033
002/070 2.00e-04  16:32:06 |   3.3699   3.3629   3.9105   0.3475 |  11.5844  11.3312  10.9799  12.6639
003/070 2.00e-04  16:27:43 |   3.4830   3.4774   4.1013   0.2777 |  16.8480  16.4492  10.9282  19.9406
004/070 2.00e-04  16:04:31 |   2.9967   2.9922   3.7089   0.2243 |   6.8617   6.8447  10.9285   0.8485
005/070 2.00e-04  15:40:06 |   2.6598   2.6558   3.5419   0.2011 |  20.9900  20.7800  10.9283  10.4961
006/070 2.00e-04  15:20:09 |   2.5658   2.5618   3.7254   0.2021 |   6.7646   6.6541  10.9280   5.5251
007/070 2.00e-0

PosixPath('artifacts/pretrained_commutative_cnn/encoder_state_v6.pt')

In [5]:
model.pretrain_history_.tail()

,epoch,train_loss,train_self_probe_loss,train_cross_probe_loss,train_lambda_cross,train_feature_alignment_loss,train_self_probe_local_loss,train_cross_probe_local_loss,train_self_probe_region_time_loss,train_cross_probe_region_time_loss,...,val_cross_probe_region_time_loss,val_self_probe_derivative_loss,val_cross_probe_derivative_loss,val_self_probe_frequency_loss,val_cross_probe_frequency_loss,val_self_probe_correlation_loss,val_cross_probe_correlation_loss,monitor_metric_raw,monitor_metric,monitor_key
53,54,0.518971,0.446317,0.451229,0.15,0.248464,0.256166,0.268237,0.027077,0.031016,...,6.551907,0.019473,0.006359,0.481473,0.295975,1.054336,0.809002,23.664724,19.380264,val_loss
54,55,0.518587,0.446637,0.445695,0.15,0.254785,0.224600,0.238891,0.058086,0.055371,...,2.095831,0.003031,0.002610,0.061416,0.061458,0.596778,0.554192,6.687475,6.687475,val_loss
55,56,0.528926,0.461919,0.413680,0.15,0.247732,0.267587,0.246683,0.034979,0.017978,...,2.521928,0.008917,0.003127,0.145779,0.087833,0.728910,0.645895,9.061837,9.061837,val_loss
56,57,0.596082,0.521762,0.465438,0.15,0.225179,0.304306,0.278773,0.054684,0.034092,...,2.094084,0.002763,0.002405,0.062648,0.063879,0.576413,0.545003,6.874095,9.061837,val_loss
57,58,0.561972,0.491027,0.442639,0.15,0.227467,0.279797,0.263965,0.050714,0.026932,...,2.510184,0.005462,0.003488,0.120296,0.080579,0.703899,0.621095,8.936974,8.936974,val_loss
